# PySpark and Spark SQL with Tables

In this notebook we will use the Boston dataset and store it the spark datawarehouse. We will then use the datawarehouse to access this data using spark sql.

In [0]:
sc = spark.sparkContext  

Display the spark object - this provides the link to the Spark UI

In [0]:
spark

SparkSession - hive 
 
 
 SparkContext 

 Spark UI 

 
 Version 
 v3.3.2 
 Master 
 local[8] 
 AppName 
 Databricks Shell

List the Spark datawarehouse. It should show the default database.

In [0]:
spark.catalog.listTables()

Out[36]: [Table(name='boston_tmp_view', catalog=None, namespace=[], description=None, tableType='TEMPORARY', isTemporary=True)]

In [0]:
df=spark.sql("show databases")
df.show()

+------------+
|databaseName|
+------------+
|     default|
|      w10_db|
+------------+



List the tables in the database. If this is a new install (you haven't run this notebook before), there shouldn't be any tables.

In [0]:
tables = spark.sql("show tables").show()

+--------+---------------+-----------+
|database|      tableName|isTemporary|
+--------+---------------+-----------+
|        |boston_tmp_view|       true|
+--------+---------------+-----------+



Now, let's load the Boston dataset into the datawarehouse. We will use the spark dataframe API to load the data. We will then use the spark sql API to create a table from the dataframe.

In [0]:
boston = spark.read.csv('/FileStore/tables/BostonHousing.csv', header=True, inferSchema=True);

# display the first 5 rows of the dataframe
boston.show(5);

+-------+----+-----+----+-----+-----+----+------+---+---+-------+-----+----+---------+
|   CRIM|  ZN|INDUS|CHAS|  NOX|   RM| AGE|   DIS|RAD|TAX|PTRATIO|LSTAT|MEDV|CAT. MEDV|
+-------+----+-----+----+-----+-----+----+------+---+---+-------+-----+----+---------+
|0.00632|18.0| 2.31|   0|0.538|6.575|65.2|  4.09|  1|296|   15.3| 4.98|24.0|        0|
|0.02731| 0.0| 7.07|   0|0.469|6.421|78.9|4.9671|  2|242|   17.8| 9.14|21.6|        0|
|0.02729| 0.0| 7.07|   0|0.469|7.185|61.1|4.9671|  2|242|   17.8| 4.03|34.7|        1|
|0.03237| 0.0| 2.18|   0|0.458|6.998|45.8|6.0622|  3|222|   18.7| 2.94|33.4|        1|
|0.06905| 0.0| 2.18|   0|0.458|7.147|54.2|6.0622|  3|222|   18.7| 5.33|36.2|        1|
+-------+----+-----+----+-----+-----+----+------+---+---+-------+-----+----+---------+
only showing top 5 rows



DataFrames can also be saved as persistent tables into Hive metastore using the saveAsTable command. Notice that an existing Hive deployment is not necessary to use this feature. Spark will create a default local Hive metastore (using Derby) for you. Unlike the createOrReplaceTempView command, saveAsTable will materialize the contents of the DataFrame and create a pointer to the data in the Hive metastore. Persistent tables will still exist even after your Spark program has restarted, as long as you maintain your connection to the same metastore. A DataFrame for a persistent table can be created by calling the table method on a SparkSession with the name of the table.

see https://spark.apache.org/docs/2.2.0/sql-programming-guide.html#saving-to-persistent-tables

We can create a table from a dataframe using the spark sql API. This will create a table in memory. The table will be lost when the spark session is terminated.

In [0]:
boston.createOrReplaceTempView("boston_tmp_view")

We can use SQL to query the table. The result is a dataframe.

In [0]:
df = spark.sql("SELECT * FROM boston_tmp_view")
df.show(5)

+-------+----+-----+----+-----+-----+----+------+---+---+-------+-----+----+---------+
|   CRIM|  ZN|INDUS|CHAS|  NOX|   RM| AGE|   DIS|RAD|TAX|PTRATIO|LSTAT|MEDV|CAT. MEDV|
+-------+----+-----+----+-----+-----+----+------+---+---+-------+-----+----+---------+
|0.00632|18.0| 2.31|   0|0.538|6.575|65.2|  4.09|  1|296|   15.3| 4.98|24.0|        0|
|0.02731| 0.0| 7.07|   0|0.469|6.421|78.9|4.9671|  2|242|   17.8| 9.14|21.6|        0|
|0.02729| 0.0| 7.07|   0|0.469|7.185|61.1|4.9671|  2|242|   17.8| 4.03|34.7|        1|
|0.03237| 0.0| 2.18|   0|0.458|6.998|45.8|6.0622|  3|222|   18.7| 2.94|33.4|        1|
|0.06905| 0.0| 2.18|   0|0.458|7.147|54.2|6.0622|  3|222|   18.7| 5.33|36.2|        1|
+-------+----+-----+----+-----+-----+----+------+---+---+-------+-----+----+---------+
only showing top 5 rows



Now, when we list the tables, we should see the boston table (and the temp table we created earlier).

In [0]:
tables = spark.sql("show tables").show()

+--------+---------------+-----------+
|database|      tableName|isTemporary|
+--------+---------------+-----------+
|        |boston_tmp_view|       true|
+--------+---------------+-----------+



We can now use the spark sql API to query the table.

In [0]:
df = spark.sql("SELECT *FROM boston_tmp_view") # note that this will generate an error
df.show()

+-------+----+-----+----+-----+-----+-----+------+---+---+-------+-----+----+---------+
|   CRIM|  ZN|INDUS|CHAS|  NOX|   RM|  AGE|   DIS|RAD|TAX|PTRATIO|LSTAT|MEDV|CAT. MEDV|
+-------+----+-----+----+-----+-----+-----+------+---+---+-------+-----+----+---------+
|0.00632|18.0| 2.31|   0|0.538|6.575| 65.2|  4.09|  1|296|   15.3| 4.98|24.0|        0|
|0.02731| 0.0| 7.07|   0|0.469|6.421| 78.9|4.9671|  2|242|   17.8| 9.14|21.6|        0|
|0.02729| 0.0| 7.07|   0|0.469|7.185| 61.1|4.9671|  2|242|   17.8| 4.03|34.7|        1|
|0.03237| 0.0| 2.18|   0|0.458|6.998| 45.8|6.0622|  3|222|   18.7| 2.94|33.4|        1|
|0.06905| 0.0| 2.18|   0|0.458|7.147| 54.2|6.0622|  3|222|   18.7| 5.33|36.2|        1|
|0.02985| 0.0| 2.18|   0|0.458| 6.43| 58.7|6.0622|  3|222|   18.7| 5.21|28.7|        0|
|0.08829|12.5| 7.87|   0|0.524|6.012| 66.6|5.5605|  5|311|   15.2|12.43|22.9|        0|
|0.14455|12.5| 7.87|   0|0.524|6.172| 96.1|5.9505|  5|311|   15.2|19.15|27.1|        0|
|0.21124|12.5| 7.87|   0|0.524|5

To save the table to the spark data-warehouse, we use the saveAsTable command. This will create a table in the spark data-warehouse.

In [0]:
type(boston)

Out[44]: pyspark.sql.dataframe.DataFrame

In [0]:
spark.sql("CREATE DATABASE IF NOT EXISTS w10_db;")

Out[45]: DataFrame[]

In [0]:
import re
from pyspark.sql.functions import col
def remove_special_characters(column_name):
    # This regex replaces anything that is not a letter or number with an empty string
    return re.sub(r'[^a-zA-Z0-9]', '', column_name)
# Assuming `df` is your Spark DataFrame
new_column_names = [remove_special_characters(col) for col in boston.columns]

# Rename the columns
boston = boston.toDF(*new_column_names)

In [0]:
#full schema to see the data types as well
boston.printSchema()

root
 |-- CRIM: double (nullable = true)
 |-- ZN: double (nullable = true)
 |-- INDUS: double (nullable = true)
 |-- CHAS: integer (nullable = true)
 |-- NOX: double (nullable = true)
 |-- RM: double (nullable = true)
 |-- AGE: double (nullable = true)
 |-- DIS: double (nullable = true)
 |-- RAD: integer (nullable = true)
 |-- TAX: integer (nullable = true)
 |-- PTRATIO: double (nullable = true)
 |-- LSTAT: double (nullable = true)
 |-- MEDV: double (nullable = true)
 |-- CATMEDV: integer (nullable = true)



In [0]:

#boston.writeTo('boston')

boston.write.mode("overwrite").saveAsTable("w10_db.boston")

#boston.write.mode("overwrite").saveAsTable("boston")


In [0]:
spark.catalog.listTables('w10_db')

Out[49]: [Table(name='boston', catalog='spark_catalog', namespace=['w10_db'], description=None, tableType='MANAGED', isTemporary=False),
 Table(name='boston_tmp_view', catalog=None, namespace=[], description=None, tableType='TEMPORARY', isTemporary=True)]

In [0]:
df = spark.sql("SELECT * FROM w10_db.boston")
df.show()

+-------+----+-----+----+-----+-----+-----+------+---+---+-------+-----+----+-------+
|   CRIM|  ZN|INDUS|CHAS|  NOX|   RM|  AGE|   DIS|RAD|TAX|PTRATIO|LSTAT|MEDV|CATMEDV|
+-------+----+-----+----+-----+-----+-----+------+---+---+-------+-----+----+-------+
|0.00632|18.0| 2.31|   0|0.538|6.575| 65.2|  4.09|  1|296|   15.3| 4.98|24.0|      0|
|0.02731| 0.0| 7.07|   0|0.469|6.421| 78.9|4.9671|  2|242|   17.8| 9.14|21.6|      0|
|0.02729| 0.0| 7.07|   0|0.469|7.185| 61.1|4.9671|  2|242|   17.8| 4.03|34.7|      1|
|0.03237| 0.0| 2.18|   0|0.458|6.998| 45.8|6.0622|  3|222|   18.7| 2.94|33.4|      1|
|0.06905| 0.0| 2.18|   0|0.458|7.147| 54.2|6.0622|  3|222|   18.7| 5.33|36.2|      1|
|0.02985| 0.0| 2.18|   0|0.458| 6.43| 58.7|6.0622|  3|222|   18.7| 5.21|28.7|      0|
|0.08829|12.5| 7.87|   0|0.524|6.012| 66.6|5.5605|  5|311|   15.2|12.43|22.9|      0|
|0.14455|12.5| 7.87|   0|0.524|6.172| 96.1|5.9505|  5|311|   15.2|19.15|27.1|      0|
|0.21124|12.5| 7.87|   0|0.524|5.631|100.0|6.0821|  5|

If we wish to drop a table from the warehouse, we can use the drop command.

In [0]:
# For now, we will keep the table and access it in another notebook. Therefore, this line is commented out
#spark.sql("DROP TABLE boston")

In [0]:
spark.catalog.listTables()

Out[52]: [Table(name='boston_tmp_view', catalog=None, namespace=[], description=None, tableType='TEMPORARY', isTemporary=True)]

In [0]:
spark.catalog.listTables('w10_db')

Out[53]: [Table(name='boston', catalog='spark_catalog', namespace=['w10_db'], description=None, tableType='MANAGED', isTemporary=False),
 Table(name='boston_tmp_view', catalog=None, namespace=[], description=None, tableType='TEMPORARY', isTemporary=True)]

In [0]:
spark.stop()